In [30]:
import json
from bs4 import BeautifulSoup
import pandas as pd

from graal.custom_types import UserName, PLFProgramName, Keyword, IntIndex
from graal.utils.text_utils import AttributionTextNormalizer
import logging
import logging.config

logging.config.fileConfig("logging.conf")


def extract_html_table_as_df(amdt_df: pd.Series) -> pd.DataFrame | None:
    # Parse the HTML content
    soup = BeautifulSoup(amdt_df["Corps amdt"], "html.parser")

    # Extract the table
    table = soup.find("table")
    if table is None:
        return None

    # Extract the table header
    header = [th.text.strip() for th in table.find_all("th")]

    # Extract rows from the table
    rows = table.find_all("tr")

    table_rows = []
    for row in rows[1:]:
        cols = row.find_all("td")
        cols = [ele.text.strip() for ele in cols]
        table_rows.append(cols)
    credit_table_df = pd.DataFrame(table_rows, columns=header)
    credit_table_df["+"] = credit_table_df["+"].fillna(0)
    credit_table_df["-"] = credit_table_df["-"].fillna(0)
    credit_table_df["+"] = credit_table_df["+"].astype(int)
    credit_table_df["-"] = credit_table_df["-"].astype(int)
    return credit_table_df


def normalize_programme_table(df: pd.DataFrame):
    df["Programmes"] = df["Programmes"].apply(
        lambda text: AttributionTextNormalizer.normalize_text(str(text))
    )
    df = df[~df["Programmes"].isin(["totaux", "solde"])]
    return df


def get_attribution_for_credit_table(
    program_to_attribution: dict[PLFProgramName, UserName],
    keyword_to_attribution: dict[Keyword, UserName],
    amdt_row: pd.Series,
    credit_table: pd.DataFrame,
) -> set[UserName]:
    normalized_expose_amdt = AttributionTextNormalizer.normalize_text(
        amdt_row["Exposé amdt"]
    )
    # There a 3 scenarios to consider, they are listed from highest to lowest priority.

    # Case 1: one of the rows for credit_table["Programmes"] contains "ligne nouvelle" or "nouveau programme"
    # If that happens, look for a keyword in normalized_expose_amdt
    logging.info("Looking at case 1 for credit table analysis")
    possible_attributions = set()
    if (
        credit_table["Programmes"]
        .str.contains("ligne nouvelle|nouveau programme")
        .any()
    ):
        for keyword, attribution in keyword_to_attribution.items():
            if keyword in normalized_expose_amdt:
                possible_attributions.add(attribution)

    if len(possible_attributions) > 0:
        return possible_attributions

    # Case 2: The integers in the + columns are all equal to 0 and there is at least one positive integer in the - columns
    # If that happens, take the names of the programs for which there is a positive integer in the - columns and look for the corresponding attribution in program_to_attribution
    # Put all possible attributions in the `possible_attributions` set
    logging.info("Looking at case 2 for credit table analysis")
    if (credit_table["+"] == 0).all() and (credit_table["-"] > 0).any():
        positive_programs = credit_table.loc[credit_table["-"] > 0, "Programmes"]
        for program in positive_programs:
            if program in program_to_attribution:
                possible_attributions.add(program_to_attribution[program])

    if len(possible_attributions) > 0:
        return possible_attributions

    # Case 3: There is at least one positive value in both the "+" and "-" columns (the positive integers don't have to be on the same row)
    # If that happens, take the names of the programs for which there is a positive integer in the "+" columns and look for the corresponding attribution in program_to_attribution
    # Put all possible attributions in the `possible_attributions` set
    logging.info("Looking at case 3 for credit table analysis")
    if (credit_table["+"] > 0).any() and (credit_table["-"] > 0).any():
        positive_programs = credit_table.loc[(credit_table["+"] > 0), "Programmes"]
        for program in positive_programs:
            if program in program_to_attribution:
                possible_attributions.add(program_to_attribution[program])

    return possible_attributions


# These 2 come from the config file
keyword_to_attribution: dict[Keyword, UserName] = {
    AttributionTextNormalizer.normalize_text("mot clé 1"): "Mister keyword 1",
    AttributionTextNormalizer.normalize_text("mot clé 2"): "Mister keyword 2",
    AttributionTextNormalizer.normalize_text("mot clé 3"): "Madam keyword 3",
}
program_to_attribution: dict[PLFProgramName, UserName] = {
    AttributionTextNormalizer.normalize_text(
        "Prévention, sécurité sanitaire et offre de soins"
    ): "Mister program A",
    AttributionTextNormalizer.normalize_text("Protection maladie"): "Madam program B",
}

print("keyword_to_attribution:")
display(keyword_to_attribution)
print("program_to_attribution:")
display(program_to_attribution)

# Load JSON file
with open("graal/exemples_corps_credits.json", "r") as file:
    amdts = json.load(file)

amendments_df = pd.DataFrame(amdts)


# This is done before the HTML is removed
def create_amdt_idx_to_credit_table(
    amendments_df: pd.DataFrame,
) -> dict[IntIndex, pd.DataFrame]:
    amdt_idx_to_credit_table: dict[IntIndex, pd.DataFrame] = {}
    for _, amdt_row in amendments_df.iterrows():
        credit_table = extract_html_table_as_df(amdt_row)
        if credit_table is not None:
            credit_table = normalize_programme_table(credit_table)
            amdt_idx_to_credit_table[amdt_row["amdt_idx"]] = credit_table
    return amdt_idx_to_credit_table


amdt_idx_to_credit_table = create_amdt_idx_to_credit_table(amendments_df)

# This is done inside the attribution block if the scenario is "PLF"
for amdt_idx, credit_table in amdt_idx_to_credit_table.items():
    amdt_row = amendments_df[amendments_df["amdt_idx"] == amdt_idx].iloc[0]
    manager = get_attribution_for_credit_table(
        program_to_attribution, keyword_to_attribution, amdt_row, credit_table
    )
    display(credit_table)
    display(manager)

keyword_to_attribution:


{'mot cle 1': 'Mister keyword 1',
 'mot cle 2': 'Mister keyword 2',
 'mot cle 3': 'Madam keyword 3'}

program_to_attribution:


{'prevention, securite sanitaire et offre de soins': 'Mister program A',
 'protection maladie': 'Madam program B'}

INFO - Looking at case 1 for credit table analysis
INFO - Looking at case 2 for credit table analysis


...

,Programmes,+,-
0,action de la france en europe et dans le monde,0,220000000
1,diplomatie culturelle et d'influence,0,0
2,francais a l'etranger et affaires consulaires,0,0
3,aide d'urgence pour les territoires palestinie...,220000000,0


{'Mister keyword 2'}

INFO - Looking at case 1 for credit table analysis
INFO - Looking at case 2 for credit table analysis
INFO - Looking at case 3 for credit table analysis


,Programmes,+,-
0,"prevention, securite sanitaire et offre de soins",10000,0
1,protection maladie,0,1249592126
2,reversement a la securite sociale des recettes...,0,0


{'Mister program A'}